# Two-Tower Recommender: Evaluation Summary

This notebook documents the evaluation of a two-tower neural retrieval model trained on a retail clickstream dataset, deployed via Databricks Feature Views, Lakebase, and Vector Search. The investigation uncovered **biased in-batch softmax** as the root cause of the user tower underperforming a simple item-mean baseline. Applying the logQ frequency correction resolved it: user-tower MRR jumped from 0.074 to 0.122, surpassing item-mean (0.110). Blending with co-occurrence via reciprocal rank fusion reaches **MRR 0.129** on a held-out test set.

On this dataset, the model does not outperform co-occurrence with a popularity fallback on accuracy alone. The case for the two-tower architecture rests on the end-to-end Feature Views demonstration and on what richer real-world features would enable — see Key Takeaways below.

All metrics use **has-history novel rows** with seen items masked, unless noted.

## Key Takeaways

**On this dataset, the model does not beat simpler approaches on accuracy alone.** Co-occurrence with a global-popularity fallback covers every evaluation slice: it matches the model on first-event rows (both reduce to popularity, MRR ≈ 0.049) and outperforms it on has-history rows (test MRR 0.120 vs 0.115). The blend (0.129) is the best measured result, but it requires the model *and* co-occurrence together.

**The model's value in this repo is as a Feature Views demonstration, not an accuracy claim.** It shows StreamSource features feeding a trained user tower end-to-end — and the user tower beats item-mean (0.122 vs 0.110 on eval), proving those features carry signal when the loss is unbiased. The logQ diagnosis itself is instructive: in-batch softmax bias is a common pitfall in retrieval systems, and the investigation shows how to detect and fix it. That is the engineering story.

### Why co-occurrence wins here: thin features

This open-source dataset has product ID, three category levels, brand, and price — nothing else. Co-occurrence directly captures "items viewed together" from session logs, and there are not enough side features for the model to learn patterns that co-occurrence misses. The two-tower model is approximating co-occurrence structure through embeddings, with no additional signal to compensate for the approximation loss.

### What would tip the balance toward a two-tower model

In a real deployment, the feature set is substantially richer. The following would give the model discriminative power that co-occurrence cannot access:

* **Product content features** — images, text descriptions, review ratings, detailed attributes (size, color, material). These let the item tower embed new products near similar ones *before* they accumulate co-view history. Co-occurrence returns nothing for items with no session overlap.
* **User-level features** — demographics, purchase history, lifetime value, subscription tier. Co-occurrence has no user representation; it treats every session identically. A user tower can learn that a high-LTV customer browsing cameras is heading toward a different price tier than a first-time visitor.
* **Request-time context (RequestSource)** — search query, referrer URL, landing page category, campaign, device, geolocation. This is what makes cold-start sessions solvable. The user tower gets signal before the first click. This dataset has none of it, which is why first-event performance equals popularity. The Feature Views architecture already supports this via `RequestSource`.
* **Cross-category discovery** — co-occurrence is strongest within-category (items co-viewed in the same session tend to be related). A model with rich features can learn cross-category patterns: someone buying a camera is likely to want a memory card, even if those items rarely appear in the same browsing session.
* **Temporal and contextual signals** — seasonality, trending items, promotional context. Co-occurrence matrices are static; a model can condition on time.
* **New product cold start** — products launched yesterday have zero co-view history. With content features, the item tower can embed them near similar existing products from day one.

### Forward-looking: RequestSource

The missing piece for cold-start improvement is context that arrives *with* the serving request — referrer, search terms, campaign. Adding even one of these via `RequestSource` would give the user tower discriminative signal on first-event rows where it currently matches popularity. This ties directly to the blog's value proposition for Feature Views: the same architecture that serves streaming session features can also inject request-time context, and the model is already structured to consume it.

## 1. Project Overview

**Goal**: Build a real-time product recommender using a two-tower neural retrieval model, deployed via Databricks Feature Views, Lakebase online store, and Vector Search.

**Dataset**: Retail e-commerce clickstream (∼25M training events, 154K products). Train: Oct 14–31. Eval: Nov 1–3 (4.6M rows). Held-out test: Nov 4–6 (5.2M rows).

**Architecture**: Shared product + brand embeddings feed into:
- **Item tower** (12 features → 64-dim embedding): product ID, 3-level category hierarchy, brand, log price, price rank, popularity, conversion rate, cart-to-purchase rate, recency
- **User tower** (12 session scalars + 4 sequence features → 64-dim embedding): session counts/values, time cyclicals, recent product/brand ID sequences
- **In-batch contrastive loss** with fixed temperature (τ = 0.05)
- 10.1M parameters (9.99M shared embeddings)

**Evaluation protocol**: Score target item against all 154K candidates, mask previously-seen items, compute MRR and Recall@k on the **has-history novel** slice (43% of eval) — sessions with browsing context where the target is new.

## 2. Evaluation Data Composition

The eval set (4,563,561 rows) breaks down into three slices:

| Slice | Count | % | Description |
| --- | --- | --- | --- |
| **Repeat** | 1,596,229 | 35.0% | Target item is in the user's recent history |
| **Novel (has history)** | 1,967,768 | 43.1% | Target is new, but user has browsing context |
| **Novel (first event)** | 999,564 | 21.9% | No history at all (cold start) |

The 35% repeat rate is critical context: over a third of events are re-views of products the user already browsed. This inflates unmasked metrics and can mislead about the model's true discovery ability. All metrics below use the **novel (has history)** slice with seen-item masking.

## 3. Baselines

We compare against two non-learned baselines:

| Baseline | Method |
| --- | --- |
| **Global popularity** | Rank candidates by training-set view count |
| **Category popularity** | Rank by view count within the user's most-viewed L1 category |

And one strong learned baseline:

| Baseline | Method |
| --- | --- |
| **Co-occurrence** | For each item in the user's history, retrieve the top-150 co-occurring items (items viewed in the same session during training), aggregate scores, rank. Built from a 154K×154K sparse co-occurrence matrix (50M non-zero entries) computed on training data only (no eval leakage). |

In [0]:
import pandas as pd

rows = [
    {"Method": "Global Popularity", "MRR": 0.0160, "Recall@10": 0.0344, "Recall@100": 0.1234},
    {"Method": "Category Popularity", "MRR": 0.0313, "Recall@10": 0.0660, "Recall@100": 0.2248},
    {"Method": "Co-occurrence (top-150)", "MRR": 0.1277, "Recall@10": 0.2783, "Recall@100": 0.6363},
]
df_baselines = pd.DataFrame(rows).set_index("Method")
display(df_baselines)

**Interpretation**: Co-occurrence is a remarkably strong baseline at 0.1277 MRR — 4× better than category popularity and 8× better than global popularity. It leverages session-level co-view patterns directly. Any neural model must beat this to justify its complexity.

**Note on tie-breaking (corrected)**: Co-occurrence scores are integer-valued, so many candidates share the same score (especially 0). The original rank formula (`rank = n_strictly_greater + 1`) favored zero-scored targets. We now use expected-rank tie-breaking: `rank = n_greater + 1 + n_tied / 2`. The correction was modest — MRR dropped 2.2% (0.1306 → 0.1277), R@10 0.8%, R@100 1.0% — confirming the main conclusions hold.

## 4. The Persistent MLP Gap

The two-tower model is evaluated with two readout strategies:

* **User tower**: The learned user embedding (MLP over 8 session scalars + 4 sequence features + pooled recent items) retrieves candidates via cosine similarity.
* **Item-mean bypass**: Skip the user tower entirely. Average the item-tower embeddings of the user's recent products and use that as the query vector.

Across every architecture and training variant tested **before the logQ fix**, item-mean consistently outperformed the user tower. The MLP was making predictions *worse* than a simple average of item embeddings:

In [0]:
rows = [
    {"Config": "v2 baseline (w=1.0)",        "Item-Mean MRR": 0.1062, "User-Tower MRR": 0.0713, "Gap": 0.0349},
    {"Config": "v2 + duplicate masking",       "Item-Mean MRR": 0.1034, "User-Tower MRR": 0.0748, "Gap": 0.0286},
    {"Config": "v3 residual + dup mask",       "Item-Mean MRR": 0.1082, "User-Tower MRR": 0.0750, "Gap": 0.0332},
    {"Config": "v4 item-pool + dup mask",      "Item-Mean MRR": 0.1067, "User-Tower MRR": 0.0746, "Gap": 0.0321},
]
df_gap = pd.DataFrame(rows).set_index("Config")
print("Has-history novel, masked — every config shows item-mean > user tower")
print(f"Co-occurrence baseline: MRR = 0.1277")
print()
display(df_gap)

## 5. Hypotheses Ruled Out

Four explanations for the MLP gap were investigated and eliminated. All fell within a 0.002–0.006 MRR noise band (measured across seeds and configurations).

**5a. Duplicate items in training batches.** In-batch negatives treat every other row as a negative. When the same product appears twice, the model gets contradictory gradients. Masking duplicates changed MRR by 0.002–0.003 — within noise.

**5b. Residual alignment (v3).** A residual connection (`user_vec = item_mean + MLP_correction`) guarantees the user tower matches item-mean at init. After training, cosine similarity between raw item-mean and item-tower output dropped to 0.54 — training destroyed the alignment. v3 MRR was indistinguishable from v2.

**5c. Gradient through pooling path (v4).** Replacing the user tower's learned pooling with a direct item-tower call routes gradients back through item embeddings. Blocking this gradient (`no_grad`) dropped item-mean MRR from 0.106 to 0.093 — confirming it matters for embedding quality — but with gradients enabled, v4 matched v2.

**5d. Train vs eval gap (overfitting).** The MLP gap was measured on both splits:
* Train (1.36M rows): item-mean 0.111, user-tower 0.084, gap 0.027
* Eval (1.97M rows): item-mean 0.104, user-tower 0.077, gap 0.028

Nearly identical. The user tower underperforms item-mean on *training data it has seen*. This is not overfitting — the objective itself is biased.

## 6. Root Cause: Biased In-Batch Softmax

In-batch negative sampling draws negatives from the mini-batch. Popular items appear as negatives far more often than rare items — in proportion to their training-set frequency. The softmax loss therefore over-penalises the model for assigning high scores to popular items, because they appear as negatives in almost every batch.

The item-mean bypass is unaffected: it uses a fixed average of embeddings and never sees the loss. But the user tower MLP is trained end-to-end against this biased loss. It learns to suppress scores for popular items — even when the session context indicates the user wants one.

**Evidence — over-correction check.** The median training-set count of the top-10 recommended items:

| | User-Tower Median | Item-Mean Median |
| --- | --- | --- |
| Without logQ | 223 | 1,008 |
| With logQ | 1,652 | 1,202 |

Without logQ, the user tower's top-10 had median popularity of 223 — far below item-mean (1,008). The user tower was actively avoiding popular items because the biased loss taught it they are bad negatives. LogQ restored natural popularity levels.

## 7. The Fix: logQ Frequency Correction

The standard fix for sampling bias in in-batch softmax is the **logQ correction** (Yi et al., 2019). For each candidate item *j* in the batch, subtract log(frequency_j) from the logit before softmax. This cancels the over-representation of popular items as negatives.

`logits[i, j] = (u_i · v_j) / τ  −  log(count_j)`

The correction is applied at training time only — at inference, the model scores all 154K items uniformly. Two seeds (42, 123) were trained to verify stability.

In [0]:
rows = [
    {"Config": "No logQ (seed 42)",  "UT MRR": 0.0740, "UT R@10": 0.1629, "UT R@100": 0.4500,
     "IM MRR": 0.1045, "IM R@10": 0.2294},
    {"Config": "logQ (seed 42)",     "UT MRR": 0.1218, "UT R@10": 0.2624, "UT R@100": 0.6125,
     "IM MRR": 0.1095, "IM R@10": 0.2388},
    {"Config": "logQ (seed 123)",    "UT MRR": 0.1213, "UT R@10": 0.2619, "UT R@100": 0.6105,
     "IM MRR": 0.1091, "IM R@10": 0.2385},
]
df_logq = pd.DataFrame(rows).set_index("Config")
print("Has-history novel, masked (Nov 1–3 eval)")
print(f"Co-occurrence baseline: MRR = 0.1277, R@10 = 0.2783, R@100 = 0.6363")
print()
display(df_logq)
print()
print("User-tower lift: MRR 0.074 → 0.122 (+0.048), now surpasses item-mean (0.110)")
print("Seed delta: 0.0004 MRR (far below 0.002–0.006 noise band from architecture changes)")
print("Remaining gap to co-occurrence: 0.006 MRR (down from 0.054)")

**Interpretation**: logQ is the single change that resolves the MLP gap. Without it, the user tower learns a popularity-suppression policy that hurts retrieval. With it, the user tower's MRR (0.122) surpasses item-mean (0.110) for the first time, confirming the MLP adds value when the loss is unbiased.

The remaining 0.006 MRR gap to co-occurrence is expected: the co-occurrence matrix captures exact session overlap counts, which embeddings can only approximate.

**Aside — repeat-weight sweep.** We also tested downweighting repeat rows (w=0.0, 0.2, 1.0). All three produced MRR in the 0.059–0.071 range (user tower) — within the noise band and all below the logQ result. w=1.0 (standard) was best. Repeat rows provide useful contrastive signal; the problem was always the biased objective, not the training data composition.

## 8. Blending with Co-occurrence (RRF)

The logQ model closes 89% of the gap to co-occurrence, but a 0.006 MRR residual remains. Since the two methods capture different signals (learned embeddings vs exact session overlap), we combine them via **reciprocal rank fusion** (RRF): `score = 1/(k + rank_model) + w × 1/(k + rank_cooc)`, k=60. We sweep w on the Nov 1–3 eval set:

In [0]:
# RRF blend: reciprocal rank fusion of model + co-occurrence
rows = [
    {"w": "ref: model alone", "MRR": 0.1218, "R@10": 0.2624, "R@100": 0.6125},
    {"w": "ref: co-occ alone", "MRR": 0.1277, "R@10": 0.2783, "R@100": 0.6363},
    {"w": "0.25",             "MRR": 0.1282, "R@10": 0.2748, "R@100": 0.6327},
    {"w": "0.50",             "MRR": 0.1309, "R@10": 0.2813, "R@100": 0.6491},
    {"w": "0.75",             "MRR": 0.1322, "R@10": 0.2852, "R@100": 0.6558},
    {"w": "1.00 (best)",      "MRR": 0.1365, "R@10": 0.2881, "R@100": 0.6578},
    {"w": "1.50",             "MRR": 0.1333, "R@10": 0.2896, "R@100": 0.6572},
    {"w": "2.00",             "MRR": 0.1331, "R@10": 0.2897, "R@100": 0.6551},
]
df_rrf = pd.DataFrame(rows).set_index("w")
print("RRF blend: score = 1/(k + rank_model) + w * 1/(k + rank_cooc), k=60")
print("Has-history novel, Nov 1–3 eval (1,967,768 rows)")
print()
display(df_rrf)
print()
print("Best w=1.0: +0.015 MRR over model alone, +0.009 over co-occ alone.")
print("Clears the 0.005 adoption bar by 3×.")

## 9. Held-Out Test: Nov 4–6

The blend weight (w=1.0) was selected on Nov 1–3 eval data. To check for overfitting to the eval window, we ran a single-pass evaluation on Nov 4–6 (5,198,815 rows, never used for tuning).

All three methods drop \~0.007 MRR uniformly — expected, since the model was trained on Oct 14–31 and the test window is further from the training distribution. The relative ordering is preserved and the blend still leads by the same margin.

### Per-slice test results (Nov 4–6)

| Slice | Rows | MRR | R@10 | R@100 |
| --- | --- | --- | --- | --- |
| Has-history novel | 2,258,883 | 0.1289 (blend) | 0.2734 | 0.6365 |
| Repeat | 1,803,267 | 0.5690 | 0.7699 | — |
| First-event (cold start) | 1,136,665 | 0.0462 | 0.1050 | — |

**Eval → test drop (has-history novel):**

| Method | Eval MRR | Test MRR | Δ |
| --- | --- | --- | --- |
| Model alone | 0.1218 | 0.1152 | −0.0066 |
| Co-occ alone | 0.1277 | 0.1204 | −0.0073 |
| Blend w=1.0 | 0.1365 | 0.1289 | −0.0076 |

The uniform drop across all methods confirms the w=1.0 selection does not overfit to the eval window.

**First-event baseline (computed)**: Global popularity is the only valid baseline for first-event rows (category popularity is undefined with no history). The model is indistinguishable from global popularity on this slice:

| | MRR | R@10 | R@100 |
| --- | --- | --- | --- |
| Global popularity (eval) | 0.0489 | 0.1069 | 0.2731 |
| Model logQ (eval) | 0.0487 | 0.1064 | 0.2768 |
| Global popularity (test) | 0.0465 | 0.1055 | 0.2647 |
| Model logQ (test) | 0.0462 | 0.1050 | — |

The deltas are −0.0002 to −0.0003 MRR — the user tower adds no value beyond popularity for cold-start sessions. The logQ fix restored the model to parity with popularity (without logQ, the model was far below it at MRR=0.005), but the user tower has no session context to leverage when there is no history.

In [0]:
print("=" * 70)
print("  FINAL NUMBERS (held-out Nov 4\u20136, has-history novel)")
print("=" * 70)
print()
print(f"  Blend (model + co-occ, w=1.0): MRR = 0.1289, R@10 = 0.2734")
print(f"  Model alone (logQ):            MRR = 0.1152, R@10 = 0.2483")
print(f"  Co-occurrence alone:           MRR = 0.1204, R@10 = 0.2633")
print(f"  Category popularity:           MRR = 0.0313, R@10 = 0.0660")
print(f"  Global popularity:             MRR = 0.0160, R@10 = 0.0344")
print()
print(f"  Blend lift over co-occ:  +0.009 MRR (+7%)")
print(f"  Blend lift over model:   +0.014 MRR (+12%)")
print(f"  Model lift over cat-pop: +0.084 MRR (3.7\u00d7)")
print()
print("  Repeat slice:      MRR = 0.569 (trivial \u2014 item is in recent history)")
print("  First-event slice: MRR = 0.046 (= global popularity; model adds no value here)")
print("=" * 70)